# Stage-2 retrain — `traffic-yolo-augmented` (Phase 8)

Trains YOLO11n on the project's stage-2 dataset (`lynkeus03/vehicle-detection-by9xs` v3, 9,211 images, 6 classes) with augmentation deliberately tuned for small/distant-vehicle detection, and logs the run to MLflow as experiment `traffic-yolo-augmented`. Uses the exact same `training/train.py` and `training/config_stage2_augmented.yaml` that live in the repo. This notebook is just the runner, no training logic is duplicated here.

**T4 GPU, ~9,200 images / batch 16 / 100 epochs**

In [ ]:
!nvidia-smi

Tue Aug 18 18:06:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Get the project code onto Colab

In [ ]:
SAVE_TO_DRIVE = True

import os

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/real-time-traffic-classifier-colab'
else:
    PROJECT_DIR = '/content/real-time-traffic-classifier-colab'

os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR
print('Working directory:', PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/real-time-traffic-classifier-colab
Working directory: /content/drive/MyDrive/real-time-traffic-classifier-colab


In [ ]:
import zipfile
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('.')
print('Extracted:', zip_name)

Saving traffic_classifier_code.zip to traffic_classifier_code.zip
Extracted: traffic_classifier_code.zip


In [ ]:
# Install project dependencies.
!pip install -q ultralytics mlflow roboflow python-dotenv pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9

## 2. Roboflow API key

Preferred path: Named it `ROBOFLOW_API_KEY`

In [ ]:
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

if not api_key:
    import getpass
    api_key = getpass.getpass('Roboflow API key (input hidden): ')

with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={api_key}\n')

api_key = None  # drop the in-memory copy once it's written to .env
print('.env written.', 'Length check only, never the value:', 'OK' if os.path.getsize('.env') > len('ROBOFLOW_API_KEY=\n') else 'EMPTY — key was blank')

Roboflow API key (input hidden): ··········
.env written. Length check only, never the value: OK


## 3. Download + prepare the stage-2 dataset

Same two-step pipeline as local training (`training/download_dataset.py` then `training/prepare_dataset.py`)

In [ ]:
!python -m training.download_dataset --workspace lynkeus03 --project vehicle-detection-by9xs --version 3

loading Roboflow workspace...
loading Roboflow project...
Project:      vehicle-detection
Public:       True
Type:         object-detection
Classes:      {'motorbike': 15074, 'bus': 6696, 'microbus': 3593, 'car': 27579, 'truck': 1559, 'pickup-van': 4277}
Splits:       {'valid': 0, 'test': 0, 'train': 9211}

Extracting Dataset Version Zip to data/raw/vehicle-detection-by9xs in yolov8:: 100% 18428/18428 [03:20<00:00, 91.83it/s]
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

Downloaded to: /content/drive/MyDrive/real-time-traffic-classifier-colab/data/raw/vehicle-detection-by9xs


In [ ]:
!python -m training.prepare_dataset \
  --source data/raw/vehicle-detection-by9xs \
  --output-dir data/processed_stage2 \
  --dataset-yaml data/dataset_stage2.yaml

train: 9211 usable images found
-> val: 921 images copied (re-split)
-> test: 921 images copied (re-split)
-> train: 7369 images copied (re-split)
Classes (6): ['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck']
Images per split: {'val': 921, 'test': 921, 'train': 7369}
Wrote data/dataset_stage2.yaml


In [ ]:
# Sanity check before committing to a long run
!cat data/dataset_stage2.yaml
!echo '---'
!echo "train images: $(ls data/processed_stage2/images/train | wc -l)"
!echo "val images:   $(ls data/processed_stage2/images/val | wc -l)"
!echo "test images:  $(ls data/processed_stage2/images/test | wc -l)"

path: /content/drive/MyDrive/real-time-traffic-classifier-colab/data/processed_stage2
train: images/train
val: images/val
test: images/test
nc: 6
names:
- bus
- car
- microbus
- motorbike
- pickup-van
- truck
---
train images: 7369
val images:   921
test images:  921


## 4. Smoke test first (2 epochs)

Before committing GPU-hours to the full 100-epoch run, run 2 epochs end-to-end to catch config/path/dependency problems. This mirrors the same local smoke test already run for the augmentation wiring itself (`--name smoke_test_aug_wiring`, see README)

In [ ]:
!python -m training.train --config training/config_stage2_augmented.yaml \
  --epochs 2 --name stage2_smoke_test --mlflow-experiment traffic-yolo-augmented-smoke

Training configuration
  data: data/dataset_stage2.yaml
  model: yolo11n.pt
  epochs: 2
  imgsz: 640
  batch: 16
  lr0: 0.01
  device: 0
  conf: 0.25
  iou: 0.45
  patience: 20
  workers: 8
  project: outputs/training_runs
  name: stage2_smoke_test
  mlflow_experiment: traffic-yolo-augmented-smoke
  scale: 0.9
  translate: 0.2
  hsv_v: 0.5
  mosaic: 1.0
  close_mosaic: 10
  copy_paste: 0.3
  copy_paste_mode: flip
2026/08/18 18:21:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/18 18:21:44 INFO mlflow.store.db.utils: Updating database tables
2026/08/18 18:21:49 INFO mlflow.tracking.fluent: Experiment with name 'traffic-yolo-augmented-smoke' does not exist. Creating a new experiment.
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_m

## 5. Full training run (100 epochs)

Training everything from `training/config_stage2_augmented.yaml` (tuned augmentation: `scale=0.9`, `translate=0.2`, `hsv_v=0.5`, `copy_paste=0.3`), logged to MLflow as `traffic-yolo-augmented`.

In [ ]:
!python -m training.train --config training/config_stage2_augmented.yaml

Training configuration
  data: data/dataset_stage2.yaml
  model: yolo11n.pt
  epochs: 100
  imgsz: 640
  batch: 16
  lr0: 0.01
  device: 0
  conf: 0.25
  iou: 0.45
  patience: 20
  workers: 8
  project: outputs/training_runs
  name: stage2_augmented
  mlflow_experiment: traffic-yolo-augmented
  scale: 0.9
  translate: 0.2
  hsv_v: 0.5
  mosaic: 1.0
  close_mosaic: 10
  copy_paste: 0.3
  copy_paste_mode: flip
2026/08/18 18:29:55 INFO mlflow.tracking.fluent: Experiment with name 'traffic-yolo-augmented' does not exist. Creating a new experiment.
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=0.25, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/dataset_stage2.yaml, degrees

## 6. Review results

Prints the same params/metrics logged to MLflow (precision/recall/mAP50/mAP50-95/FPS/latency), read straight from the run.

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
client = mlflow.MlflowClient()
exp = client.get_experiment_by_name('traffic-yolo-augmented')
runs = client.search_runs(exp.experiment_id, order_by=['start_time DESC'], max_results=1)
run = runs[0]

print('Run:', run.info.run_id, '-', run.info.status)
print('\nParams:')
for k, v in sorted(run.data.params.items()):
    print(f'  {k}: {v}')
print('\nMetrics:')
for k, v in sorted(run.data.metrics.items()):
    print(f'  {k}: {v:.4f}')

Run: 0b771b98525145ecbd87073a13618fa4 - FINISHED

Params:
  agnostic_nms: False
  amp: True
  angle: 1.0
  aug_close_mosaic: 10
  aug_copy_paste: 0.3
  aug_copy_paste_mode: flip
  aug_hsv_v: 0.5
  aug_mosaic: 1.0
  aug_scale: 0.9
  aug_translate: 0.2
  augment: False
  auto_augment: randaugment
  batch: 16
  bgr: 0.0
  box: 7.5
  cache: False
  cfg: None
  channels_last: False
  classes: None
  close_mosaic: 10
  cls: 0.5
  cls_pw: 0.0
  cls_remap: True
  compile: False
  conf: 0.25
  copy_paste: 0.3
  copy_paste_mode: flip
  cos_lr: False
  cutmix: 0.0
  data: data/dataset_stage2.yaml
  dataset: data/dataset_stage2.yaml
  degrees: 0.0
  deterministic: True
  device: 0
  dfl: 1.5
  dgrad: 0.5
  dis: 6.0
  distill_model: None
  dlam: 1.0
  dlog: 1.0
  dnn: False
  dropout: 0.0
  dynamic: False
  embed: None
  end2end: None
  epochs: 100
  erasing: 0.4
  exist_ok: True
  fliplr: 0.5
  flipud: 0.0
  format: torchscript
  fraction: 1.0
  freeze: None
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.

## 7. Get the weights back

Zips `best.pt` plus the MLflow tracking db (so the run's params/metrics travel with the weights, not just the weights alone).

In [ ]:
import shutil
from pathlib import Path

best_pt = Path('outputs/training_runs/stage2_augmented/weights/best.pt')
assert best_pt.exists(), f'{best_pt} not found — did the full training run (section 5) complete?'

bundle_dir = Path('stage2_augmented_bundle')
bundle_dir.mkdir(exist_ok=True)
shutil.copy2(best_pt, bundle_dir / 'best.pt')
shutil.copy2('mlflow.db', bundle_dir / 'mlflow.db')

archive = shutil.make_archive('stage2_augmented_bundle', 'zip', bundle_dir)
print('Bundle ready:', archive)

from google.colab import files
files.download(archive)

Bundle ready: /content/drive/MyDrive/real-time-traffic-classifier-colab/stage2_augmented_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>